In [9]:
from core.IdlParser import IdlParser
from core.Idlexer import IdlLexer

from antlr4 import FileStream, CommonTokenStream

In [88]:
path = '../testschema.avdl'

input_stream = FileStream(path)
lexer = IdlLexer(input_stream)
tokens = CommonTokenStream(lexer)
parser = IdlParser(tokens)

tree = parser.idlFile()

In [89]:
tree.importStatement()

In [90]:
import_type = tree.importStatement(2).importType
import_location = tree.importStatement(2).location

In [91]:
import_type.text

'idl'

In [92]:
import_location.text.strip('"').rsplit('.', 1)

['barack.alma', 'avsc']

In [93]:
namespace_declaration = tree.namespaceDeclaration()
namespace_declaration

In [94]:
main_namespace = namespace_declaration.namespace.getText()
main_namespace

'test.schema'

In [95]:
main_schema_declaration = tree.mainSchemaDeclaration()
main_schema_declaration

In [96]:
main_schema_type_context = main_schema_declaration.mainSchema
main_schema_type_context

In [97]:
main_schema_type_context.plainType().getText()

'TestSchemaaaaaa'

In [98]:
first_schema = tree.namedSchemaDeclaration(0)
first_schema

In [99]:
first_record = first_schema.recordDeclaration()
first_record

In [100]:
first_record_first_prop = first_record.schemaProperty(0)
first_record_first_prop

In [101]:
first_record_first_prop.name.getText()

AttributeError: 'NoneType' object has no attribute 'name'

In [85]:
first_record_first_prop.value.getText()

'{"a":true,"b":null,"c":[1,2,3,4]}'

In [87]:
first_record_first_prop.jsonValue().getText()

'{"a":true,"b":null,"c":[1,2,3,4]}'

In [27]:
%pip install avro

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://images.cicd.erste.hu/repository/pypi/simple
  Using cached https://images.cicd.erste.hu/repository/pypi/packages/avro/1.12.1/avro-1.12.1-py2.py3-none-any.whl (124 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from avro import schema

<table>
<th>
<td>Avro type</td>
<td>JSON type</td>
<td>Java type</td></th>
<tr>
<td><code>null</code></td>
<td><code>null</code></td>
<td>{@link #NULL_VALUE}</td>
</tr>
<tr>
<td><code>boolean</code></td>
<td>Boolean</td>
<td><code>boolean</code></td>
</tr>
<tr>
<td><code>int</code></td>
<td>Number</td>
<td><code>int</code></td>
</tr>
<tr>
<td><code>long</code></td>
<td>Number</td>
<td><code>long</code></td>
</tr>
<tr>
<td><code>float</code></td>
<td>Number</td>
<td><code>float</code></td>
</tr>
<tr>
<td><code>double</code></td>
<td>Number</td>
<td><code>double</code></td>
</tr>
<tr>
<td><code>bytes</code></td>
<td>String</td>
<td><code>byte[]</code></td>
</tr>
<tr>
<td><code>string</code></td>
<td>String</td>
<td>{@link java.lang.String}</td>
</tr>
<tr>
<td><code>record</code></td>
<td>Object</td>
<td>{@link java.util.Map}</td>
</tr>
<tr>
<td><code>enum</code></td>
<td>String</td>
<td>{@link java.lang.String}</td>
</tr>
<tr>
<td><code>array</code></td>
<td>Array</td>
<td>{@link java.util.Collection}</td>
</tr>
<tr>
<td><code>map</code></td>
<td>Object</td>
<td>{@link java.util.Map}</td>
</tr>
<tr>
<td><code>fixed</code></td>
<td>String</td>
<td><code>byte[]</code></td>
</tr>
</table>

# Test listener

In [4]:
!pip install antlr4-python3-runtime

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://images.cicd.erste.hu/repository/pypi/simple


In [5]:
from core.IdlParser import IdlParser
from core.IdlListener import IdlListener
from core.IdlLexer import IdlLexer

from typing import Deque
from antlr4 import FileStream, CommonTokenStream, Token, ParseTreeWalker

In [10]:
class JsonNull:
    __str__ = lambda self: "null"

JSON_NULL = JsonNull()

class TetsListener(IdlListener):
    def __init__(self):
        super().__init__()
        self.json_values = Deque()

    def exitJsonValue(self, ctx: IdlParser.JsonValueContext):
        if isinstance(ctx.parentCtx, IdlParser.JsonArrayContext):
            value = self.json_values.popleft()
            assert isinstance(self.json_values[0], list)
            self.json_values[0].append(value)

    def exitJsonLiteral(self, ctx: IdlParser.JsonLiteralContext):
        literal: Token = ctx.literal
        match literal.type:
            case IdlParser.Null:
                self.json_values.appendleft(JSON_NULL)
            case IdlParser.BTrue:
                self.json_values.appendleft(True)
            case IdlParser.BFalse:
                self.json_values.appendleft(False)
            case IdlParser.IntegerLiteral:
                number: str = literal.text.replace("_", "")
                last_char = number[-1]
                if (last_char in 'lL'):
                    number = number[:-1]
                int_number = int(number)
                self.json_values.appendleft(int_number)
            case IdlParser.FloatingPointLiteral:
                self.json_values.appendleft(float(literal.text))
            case _:
                self.json_values.appendleft(self.get_string(literal))

    def enterJsonArray(self, ctx: IdlParser.JsonArrayContext):
        self.json_values.appendleft([])

    def enterJsonObject(self, ctx: IdlParser.JsonObjectContext):
        self.json_values.appendleft({})

    def exitJsonPair(self, ctx: IdlParser.JsonPairContext):
        name: str = self.get_string(ctx.name)
        value = self.json_values.popleft()
        assert isinstance(self.json_values[0], dict)
        self.json_values[0][name] = value

    def get_string(self, string_token: Token):
        string_literal = string_token.text
        return string_literal[1:-1]
    

In [14]:
path = '../testschema.avdl'

input_stream = FileStream(path)
lexer = IdlLexer(input_stream)
tokens = CommonTokenStream(lexer)
parser = IdlParser(tokens)

tree = parser.idlFile()

In [15]:
walker = ParseTreeWalker()
listener = TetsListener()
walker.walk(listener, tree)

In [16]:
listener.json_values

deque([{'some_prop': <__main__.JsonNull at 0x22f55e04980>,
        'other_prop': [1, 2, 3]},
       'string',
       'alma',
       'alma',
       {'a': True,
        'b': <__main__.JsonNull at 0x22f55e04980>,
        'c': [1, 2, 3, 4],
        'd': 'DDDD'}])

In [ ]:
class ParseContext:
    def __init__(self):
        self.parse_context = "smth"
        self.l = []
    def add_element(self, element):
        self.l.append(element)

class IdlReader:
    def __init__(self, parse_context = None):
        self.parse_context = parse_context or ParseContext()

In [14]:
ir1 = IdlReader()
ir2 = IdlReader()
ir3 = IdlReader(ParseContext())
ir1.parse_context.add_element(1)
ir3.parse_context.add_element(3)
print(ir1.parse_context.l)
print(ir2.parse_context.l)
print(ir3.parse_context.l)

[1]
[]
[3]


In [ ]:
class IdlReader:
    def __init__(self):
        self.parse_context = "something"
    
    class IdlParserListener(IdlListener):
        def __init__(self):
            pass
        def exitIdlFile(self, ctx):
            self.parse_contex # In java it could access non-static private final parse_context of the containing class

In [4]:
class JsonProperties:
    def __init__(self):
        self.props = {}
    def __iter__(self):
        return iter(self.props.items())

In [5]:
jp = JsonProperties()
jp.props["a"] = 2
jp.props["d"] = 99
jp.props["b"] = 0

In [ ]:
for key, field_name in jp:
    print(key, field_name)

a 2
d 99
b 0


In [8]:
{"a": 1, "b": [0,2]} == {"b": [2,0], "a": 1}

False

In [1]:
from enum import Enum

In [2]:
class Type(Enum):
	INT = "int"
	BOOL = "bool"
	
	def __contains__(value):
		if isinstance(value, Type):
			return True
		for _type in Type:
			if value == _type.value:
				return True
		return False

In [8]:
Type.INT in Type

True

In [12]:
set([
    "Generator",
    "Generator",
    "Generator",
    "gen.useDefaultPrettyPrinter",
    "gen.flush",
    "Generator",
    "gen.writeString",
    "gen.writeStartObject",
    "gen.writeStringField",
    "gen.writeEndObject",
    "Generator",
    "Generator",
    "gen.writeStringField",
    "gen.writeStringField",
    "gen.writeStringField",
    "Generator",
    "gen.writeString",
    "Generator",
    "Generator",
    "gen.writeFieldName",
    "gen.writeStartArray",
    "gen.writeString",
    "gen.writeEndArray",
    "Generator",
    "gen.writeStartObject",
    "gen.writeStringField",
    "gen.writeStringField",
    "gen.writeFieldName",
    "gen.writeEndObject",
    "Generator",
    "gen.writeStartArray",
    "gen.writeStartObject",
    "gen.writeStringField",
    "gen.writeFieldName",
    "gen.writeStringField",
    "gen.writeFieldName",
    "gen.writeTree",
    "gen.writeStringField",
    "gen.writeFieldName",
    "gen.writeStartArray",
    "gen.writeString",
    "gen.writeEndArray",
    "gen.writeEndObject",
    "gen.writeEndArray",
    "Generator",
    "gen.writeStartObject",
    "gen.writeStringField",
    "gen.writeStringField",
    "gen.writeArrayFieldStart",
    "gen.writeString",
    "gen.writeEndArray",
    "gen.writeStringField",
    "gen.writeEndObject",
    "Generator",
    "gen.writeStartObject",
    "gen.writeStringField",
    "gen.writeFieldName",
    "gen.writeEndObject",
    "Generator",
    "gen.writeStartObject",
    "gen.writeStringField",
    "gen.writeFieldName",
    "gen.writeEndObject",
    "Generator",
    "gen.writeStartArray",
    "gen.writeEndArray",
    "Generator",
    "gen.writeStartObject",
    "gen.writeStringField",
    "gen.writeStringField",
    "gen.writeNumberField",
    "gen.writeEndObject"
])

{'Generator',
 'gen.flush',
 'gen.useDefaultPrettyPrinter',
 'gen.writeArrayFieldStart',
 'gen.writeEndArray',
 'gen.writeEndObject',
 'gen.writeFieldName',
 'gen.writeNumberField',
 'gen.writeStartArray',
 'gen.writeStartObject',
 'gen.writeString',
 'gen.writeStringField',
 'gen.writeTree'}

In [1]:
import json

In [14]:
from typing import Literal, Deque

In [5]:
encoder = json.JSONEncoder()

In [12]:
a = []
b = [a]
a.append(b)

DONE:
- gen.writeStartArray 
- gen.writeEndArray 
- gen.writeStartObject 
- gen.writeEndObject 
- gen.writeFieldName 
- gen.writeStringField
- gen.writeString 
- gen.writeTree
- gen.writeNumberField 
- gen.writeArrayFieldStart 

TO BE DONE:

In [22]:
from typing import Union

In [26]:
encoder.default({})

TypeError: Object of type dict is not JSON serializable

In [23]:
encoder.encode(["alma", {"a": 2.3, "b": True}])

'["alma", {"a": 2.3, "b": true}]'

In [ ]:
encoder.

In [47]:
from json import JSONEncoder
from typing import Union

class JsonGenerator:
    def __init__(self):
        self.complete = False
        self.initialized = False
        self.result = None
        self.current = []
        self.field_name = None
        self.encoder = JSONEncoder()

    def __str__(self):
        if not self.initialized:
            raise ValueError("Cannot convert empty json")
        if self.field_name is not None:
            raise ValueError("Cannot convert json, as the last object value is not set.")
        return self.encoder.encode(self.result)

    @property
    def current_container(self):
        if len(self.current) != 0:
            return self.current[-1]
        else:
            return None

    def write_start_array(self):
        array = []
        if not self.init_result(array):
            self.write(array)
        self.current.append(array)

    def write_end_array(self):
        if not isinstance(self.current_container, list):
            raise ValueError("Cannot end an array, as its not started.")
        self.current.pop()

    def write_start_object(self):
        _object = {}
        if not self.init_result(_object):
            self.write(_object)
        self.current.append(_object)

    def write_end_object(self):
        if not isinstance(self.current_container, dict):
            raise ValueError("Cannot end an object, as its not started.")
        self.current.pop()

    def write_field_name(self, field_name: str):
        if not isinstance(field_name, str):
            raise ValueError("Json object field name must be a string")
        if not isinstance(self.current_container, dict):
            raise ValueError("Cannot write json object field name. Start a json object first")
        if self.field_name is not None:
            raise ValueError("Cannot write json object field name, as the previous field value did not set.")
        self.field_name = field_name

    def write_tree(self, tree):
        self.write(tree)

    def write_string(self, value: str):
        if not isinstance(value, str):
            raise ValueError(f"Cannot write string value of type: {type(value)}")
        self.write(value)
        
    def write_number(self, value: Union[int, float]):
        if not isinstance(value, int) or not isinstance(value, float):
            raise ValueError(f"Cannot write number value of type: {type(value)}")
        self.write(value)
    
    def write_number_field(self, field_name: str, value: Union[int, float]):
        self.write_field_name(field_name)
        self.write_number(value)
        
    def write_string_field(self, field_name: str, value: str):
        self.write_field_name(field_name)
        self.write_string(value)

    def write_array_field_start(self, field_name: str):
        self.write_field_name(field_name)
        self.write_start_array()

    def write_object_field(self, field_name, value):
        self.write_field_name(field_name)
        self.write(value)
    
    def write(self, value):
        if self.complete:
            raise ValueError("Cannot write to json, as its complete")
        if isinstance(self.current_container, dict):
            if self.field_name is None:
                raise ValueError("Cannot write element to json object as no name was specified.")
            self.current_container[self.field_name] = self.validate_value(value)
            self.field_name = None
        elif isinstance(self.current_container, list):
            self.current_container.append(self.validate_value(value))
        else:
            self.result = value
            self.initialized = True
            self.complete = True

    def validate_value(self, value):
        try:
            self.encoder.encode(value)
            return value
        except Exception as e:
            raise e
        
    def init_result(self, element):
        if not self.initialized:
            self.result = element
            self.initialized = True
            return True
        return False

In [48]:
gen = JsonGenerator()
gen.write_string("alma")
str(gen)

'"alma"'

In [49]:
gen = JsonGenerator()
gen.write_start_object()
gen.write_string_field("name", "MyRecord")
gen.write_field_name("namespace")
gen.write_string("hu.erste")
gen.write_array_field_start("fields")
gen.write_start_object()
gen.write_string_field("name", "field1")
gen.write_string_field("type", "string")
gen.write_field_name("aliases")
gen.write_start_array()
gen.write_string("alma")
gen.write_end_array()
gen.write_end_object()
gen.write_end_array()
gen.write_end_object()
str(gen)

'{"name": "MyRecord", "namespace": "hu.erste", "fields": [{"name": "field1", "type": "string", "aliases": ["alma"]}]}'

In [18]:
from typing import Deque

d = Deque()
d.appendleft(1)
d.appendleft(2)
d.appendleft(3)

assert d[-1] == d.popleft()

AssertionError: 